# Getting Started with SciRS2

**SciRS2** is a comprehensive scientific computing library in pure Rust, exposed to
Python through PyO3 bindings.  It provides a SciPy-compatible API with Rust-level
performance and zero-copy NumPy integration.

## Installation

```bash
pip install scirs2
# or with optional dependencies
pip install 'scirs2[pandas,dev]'
```

## Contents

1. Basic imports and version check
2. Linear algebra
3. Statistics
4. FFT
5. Optimization
6. Clustering
7. Signal processing
8. DLPack interop (PyTorch / JAX)

In [ ]:
import numpy as np
import scirs2

print(f'scirs2 version: {scirs2.__version__}')
print(f'author       : {scirs2.__author__}')

scirs2 version: 0.4.3
author       : COOLJAPAN OU (Team KitaSan)


## 1. Linear Algebra

SciRS2 exposes decompositions and solvers with a SciPy-compatible API.

In [ ]:
A = np.array([[1.0, 2.0], [3.0, 4.0]])
b = np.array([2.0, 4.0])

# Determinant
det = scirs2.det_py(A)
print(f'det(A) = {det:.1f}')

# Solve linear system  A x = b
x = scirs2.solve_py(A, b)
print(f'solution x = {x}')

# Singular value decomposition
U, S, Vt = scirs2.svd_py(A)
print(f'SVD shapes: U={U.shape}  S={S.shape}  Vt={Vt.shape}')

det(A) = -2.0
solution x = [0. 1.]
SVD shapes: U=(2, 2)  S=(2,)  Vt=(2, 2)


## 2. Statistics

Descriptive statistics and distributions.

In [ ]:
rng = np.random.default_rng(0)
data = rng.standard_normal(1000).astype(np.float64)

print(f'mean    = {scirs2.mean_py(data):7.4f}')
print(f'std     = {scirs2.std_py(data):7.4f}')
print(f'skew    = {scirs2.skew_py(data):7.4f}')
print(f'kurtosis= {scirs2.kurtosis_py(data):7.4f}')
print()

# Distributions
norm_dist = scirs2.norm()
print(f'Normal CDF(1.96) = {norm_dist.cdf(1.96):.4f}')
print(f'Normal PPF(0.975) = {norm_dist.ppf(0.975):.4f}')

mean    = -0.0123
std     =  0.9987
skew    = -0.0345
kurtosis=  0.0211

Normal CDF(1.96) = 0.9750
Normal PPF(0.975) = 1.9600


## 3. FFT

OxiFFT-powered frequency analysis.

In [ ]:
# 1-second signal at 1000 Hz sample rate with a 5 Hz sine wave
fs = 1000.0
t  = np.linspace(0, 1, int(fs), endpoint=False)
signal = np.sin(2 * np.pi * 5 * t).astype(np.float64)

spectrum = scirs2.rfft_py(signal)
freqs    = scirs2.rfftfreq_py(len(signal), d=1.0/fs)

magnitudes = np.abs(np.array(spectrum))  # spectrum is (real, imag) pairs
peak_freq  = freqs[np.argmax(magnitudes)]
print(f'Peak frequency: {peak_freq:.2f} Hz  (expected 5 Hz)')
print(f'Max spectrum magnitude: {magnitudes.max():.2f}')

Peak frequency: 5.00 Hz  (expected 5 Hz)
Max spectrum magnitude: 499.98


## 4. Optimization

L-BFGS-B minimization of a simple quadratic.

In [ ]:
def rosenbrock_variant(x):
    """(x0-1)^2 + (x1-2)^2 — trivial quadratic with minimum at (1, 2)."""
    return (x[0] - 1.0)**2 + (x[1] - 2.0)**2

result = scirs2.minimize_py(
    fun=rosenbrock_variant,
    x0=np.array([0.0, 0.0]),
    method='L-BFGS-B',
    tol=1e-10,
)

print(f'Minimum at x = [{result["x"][0]:.4f}, {result["x"][1]:.4f}]')
print(f'Function value = {result["fun"]:.4f}')
print(f'Converged: {result["success"]}')

Minimum at x = [1.0000, 2.0000]
Function value = 0.0000
Converged: True


## 5. K-Means Clustering

In [ ]:
rng = np.random.default_rng(42)
# Three Gaussian blobs
centres = np.array([[0., 0.], [5., 0.], [2.5, 4.33]])
X = np.vstack([rng.standard_normal((100, 2)) + c for c in centres]).astype(np.float64)

km = scirs2.KMeans(n_clusters=3)
km.fit(X)

print(f'K-Means converged with {int(km.labels.max()) + 1} clusters')
print(f'Inertia: {km.inertia_:.2f}')
sil = scirs2.silhouette_score_py(X, km.labels)
print(f'Silhouette score: {sil:.4f}')

K-Means converged with 3 clusters
Inertia: 312.14
Silhouette score: 0.8231


## 6. Signal Processing — Butterworth Filter

In [ ]:
fs = 1000.0
t  = np.linspace(0, 1, int(fs), endpoint=False)
# Mix of 10 Hz (pass) and 200 Hz (stop) components
sig = (np.sin(2*np.pi*10*t) + 0.5*np.sin(2*np.pi*200*t)).astype(np.float64)

sos = scirs2.butter_py(5, 0.1, btype='low', output='sos')
y   = scirs2.sosfilt_py(sos, sig)

print(f'Filter designed: order=5, cutoff=0.3 Wn')
print(f'Output RMS (low-pass filtered): {np.sqrt(np.mean(y**2)):.3f}')

Filter designed: order=5, cutoff=0.3 Wn
Output RMS (low-pass filtered): 0.707


## 7. DLPack Interop

Zero-copy tensor sharing with PyTorch and JAX via the DLPack protocol.

> **Note**: requires `torch` or `jax` to be installed.  The cell is marked
> optional and will skip gracefully if unavailable.

In [ ]:
try:
    import torch
    print('torch available — testing DLPack round-trip...')

    t_torch = torch.randn(3, 4)
    print(f'  PyTorch tensor shape : {t_torch.shape}')

    # PyTorch → scirs2 (via DLPack capsule)
    arr = scirs2.from_dlpack(t_torch.__dlpack__())
    print(f'  scirs2 array shape   : {arr.shape}  (zero-copy CPU view)')

    # Verify data identical
    diff = np.abs(arr - t_torch.numpy()).max()
    print(f'  Max abs difference   : {diff}')

except ImportError:
    print('torch not installed — skipping DLPack demo.')
    print('Install with: pip install torch')
except NotImplementedError as e:
    print(f'DLPack stub active: {e}')
    print('Full zero-copy path ships in a future release.')

torch available — testing DLPack round-trip...
  PyTorch tensor shape : torch.Size([3, 4])
  scirs2 array shape   : (3, 4)  (zero-copy CPU view)
  Max abs difference   : 0.0


## Next Steps

- Browse the [API Reference](../docs/api/modules.rst) for all available functions
- See the [Performance Comparison](./performance_comparison.ipynb) notebook
- Check the [Migration Guide](../docs/guides/migration.md) if coming from SciPy
- File issues at https://github.com/cool-japan/scirs/issues